# Twitter regression embeddings in 2D

Load the prepared EmbeddingGemma 2 vectors, optionally tune a target-shaped IMSTE layout, review and override the suggested settings, then compare UMAP and IMSTE. The numeric target is log retweet count; its values influence graph-edge attraction during fitting.

Run notebooks/dataset_setup/prepare_twitter_dataset.ipynb first. The plot can be colored by source batch, language, posting time, retweet status or count, hashtag, or engagement.

## 1. Load the prepared vectors

In [1]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = 'retina'
from pathlib import Path
import sys, time
import numpy as np
import pandas as pd
from IPython.display import display
PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "pyproject.toml").is_file() and (p / "imste").is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Could not locate the IMSTE repository.")
for path in (PROJECT_ROOT, PROJECT_ROOT / "notebooks" / "src"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
import imste
if not Path(imste.__file__).resolve().is_relative_to(PROJECT_ROOT.resolve()):
    raise RuntimeError("Restart the kernel to load IMSTE from this checkout.")
from twitter_dataset import load_twitter_dataset
DATA_DIR = PROJECT_ROOT / "data" / "twitter"
tweets_all, X_all, data_manifest = load_twitter_dataset(DATA_DIR)
print(f"Loaded {len(tweets_all):,} tweets × {X_all.shape[1]:,} features")
print("Embedding model:", data_manifest["embedding_model"])
print("Dataset ID:", data_manifest["dataset_id"])

Loaded 100,000 tweets × 768 features
Embedding model: google/embeddinggemma-2
Dataset ID: 6832b7a1aa53862b5bc2


## 2. Choose layout sample and point colors

Set a cap for a quick first look. The selected subset is deterministic.

In [2]:
MAX_TWEETS_FOR_LAYOUT = None  # Set None to use all cached tweets
COLOR_BY = "language"      # source_batch, language, hour, weekday, retweet, retweet_count, hashtag, engagement
TOP_HASHTAGS = 12
RANDOM_STATE = int(data_manifest["random_state"])
tweets = tweets_all.copy()
X = X_all
if MAX_TWEETS_FOR_LAYOUT is not None:
    if not isinstance(MAX_TWEETS_FOR_LAYOUT, int) or MAX_TWEETS_FOR_LAYOUT < 2:
        raise ValueError("MAX_TWEETS_FOR_LAYOUT must be at least 2 or None.")
    if len(tweets) > MAX_TWEETS_FOR_LAYOUT:
        selected = np.sort(np.random.default_rng(RANDOM_STATE).choice(
            len(tweets), size=MAX_TWEETS_FOR_LAYOUT, replace=False))
        tweets = tweets.iloc[selected].reset_index(drop=True)
        X = X[selected]

retweet_counts = pd.to_numeric(tweets["retweet_count"], errors="coerce").fillna(0).clip(lower=0)
REGRESSION_TARGET_NAME = "log1p(retweet_count)"
regression_target = np.log1p(retweet_counts.to_numpy(dtype=np.float64))
TARGET_HAS_VARIATION = np.ptp(regression_target) > 0
if not TARGET_HAS_VARIATION:
    print("Retweet counts are constant; regression tuning and target shaping are disabled for this sample.")

timestamps = pd.to_datetime(tweets["created_at"], errors="coerce", utc=True)
if COLOR_BY == "source_batch":
    color_values = tweets["source_batch"].fillna("unknown").astype(str)
elif COLOR_BY == "language":
    color_values = tweets["lang"].fillna("unknown").astype(str)
elif COLOR_BY == "hour":
    color_values = timestamps.dt.hour.map(lambda x: f"{int(x):02d}:00" if pd.notna(x) else "unknown")
elif COLOR_BY == "weekday":
    color_values = timestamps.dt.day_name().fillna("unknown")
elif COLOR_BY == "retweet":
    color_values = tweets["is_retweet"].map({True: "retweet", False: "original"}).fillna("unknown")
elif COLOR_BY == "retweet_count":
    color_values = pd.cut(
        retweet_counts,
        bins=[-0.5, 0.5, 1.5, 5.5, 20.5, 100.5, np.inf],
        labels=["0", "1", "2–5", "6–20", "21–100", ">100"],
    ).astype(str)
elif COLOR_BY == "hashtag":
    counts = tweets["top_hashtag"].value_counts()
    top = set(counts.head(TOP_HASHTAGS).index)
    color_values = tweets["top_hashtag"].where(tweets["top_hashtag"].isin(top), "other / no hashtag")
elif COLOR_BY == "engagement":
    favorites = pd.to_numeric(tweets["favorite_count"], errors="coerce").fillna(0).clip(lower=0)
    engagement = retweet_counts + favorites
    if engagement.nunique() < 2:
        color_values = pd.Series(f"all tweets: {engagement.iloc[0]:g} engagement", index=tweets.index, dtype="string")
    else:
        color_values = pd.qcut(np.log1p(engagement), q=5, duplicates="drop").astype(str)
else:
    raise ValueError("Unknown COLOR_BY choice.")
counts = color_values.astype(str).value_counts()
if COLOR_BY in {"source_batch", "language", "hashtag"} and len(counts) > 20:
    keep = set(counts.head(19).index)
    color_values = color_values.astype(str).where(color_values.astype(str).isin(keep), "other")
labels = pd.Categorical(color_values.astype(str))
print(f"Using {len(tweets):,} tweets; target={REGRESSION_TARGET_NAME}; coloring by {COLOR_BY}.")
display(pd.Series(labels.astype(str)).value_counts().head(20).rename("tweets").to_frame())

Retweet counts are constant; regression tuning and target shaping are disabled for this sample.
Using 100,000 tweets; target=log1p(retweet_count); coloring by language.


/var/folders/wr/3lmv1jtx3ng07d2c77xprblh0000gp/T/ipykernel_21033/2311845587.py:23: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  timestamps = pd.to_datetime(tweets["created_at"], errors="coerce", utc=True)


,tweets
en,34164
pt,20358
es,20336
und,5797
tr,3614
fr,3208
ja,2519
in,2413
ar,1608
other,1479


## 3. Configure UMAP and tune IMSTE

In [3]:
import umap
from imste import IteratedMinimumSpanningTreeEmbedder
METRIC = "euclidean"
UMAP_SETTINGS = {
    "n_components": 2, "n_neighbors": 15, "min_dist": 0.1,
    "metric": METRIC, "random_state": RANDOM_STATE,
    "init": "recursive", "recursive_coarsening_ratio": 4,
    "compatibility_layout": False, "nearest_neighbor_backend": "rinnd",
    "n_epochs": 200, "verbose": True,
}
IMSTE_BASE_PARAMS = {
    "n_msts": 3, "learning_rate": 0.05, "negative_ratio": 5,
    "n_epochs": 200, "target_strength": 1.0, "target_bandwidth": 1.0,
    "gravity_strength": 0.2, "mean_shift_iterations": 3,
}
RUN_IMSTE_TUNING = False
TUNE_MAX_PER_BATCH = 150
TUNE_EPOCHS = 60
TUNE_TRIALS = 20
TUNE_SEARCH_SPACE = {
    "n_msts": (2, 32), "negative_ratio": (1, 10),
    "logistic_margin": (0.25, 2.0), "logistic_temperature": (0.1, 1.0),
    "lambda_rep": (0.25, 0.75), "gravity_strength": (0.1, 1.0),
    "gravity_resolution": (0.5, 2.0), "gravity_min_dist": (0.01, 1.0),
    "mean_shift_iterations": (2, 5), "include_knn_attraction": [False, True],
    "target_strength": (0.0, 4.0), "target_bandwidth": (0.25, 2.0),
}

In [4]:
tuning_seconds = 0.0
IMSTE_TUNED_PARAMS = {}
if RUN_IMSTE_TUNING and TARGET_HAS_VARIATION:
    rng = np.random.default_rng(RANDOM_STATE)
    batch_values = tweets["source_batch"].to_numpy()
    tune_indices = []
    for batch in pd.unique(batch_values):
        batch_indices = np.flatnonzero(batch_values == batch)
        n = min(len(batch_indices), TUNE_MAX_PER_BATCH)
        tune_indices.append(rng.choice(batch_indices, size=n, replace=False))
    tune_indices = np.sort(np.concatenate(tune_indices))
    X_tune, target_tune = X[tune_indices], regression_target[tune_indices]
    print(f"Tuning on {len(tune_indices):,} tweets from {tweets['source_batch'].nunique()} batches")
    tune_mapper = IteratedMinimumSpanningTreeEmbedder(
        **{**IMSTE_BASE_PARAMS, "n_epochs": TUNE_EPOCHS,
           "random_state": RANDOM_STATE, "metric": METRIC, "verbose": True}
    )
    started = time.perf_counter()
    tune_result = tune_mapper.tune(
        X_tune, target_tune, task="regression", search_space=TUNE_SEARCH_SPACE,
        n_samples=TUNE_TRIALS, n_initial_points=min(10, TUNE_TRIALS),
        coverage_weight=0.4,
    )
    tuning_seconds = time.perf_counter() - started
    IMSTE_TUNED_PARAMS = dict(tune_result.best_params)
    print(
        f"Best CV R²: {tune_result.best_r2:.3f}; MAE: {tune_result.best_mae:.3f}; "
        f"Spearman: {tune_result.best_spearman:.3f}; coverage: {tune_result.best_coverage:.3f}"
    )
    trials = pd.DataFrame(tune_result.x_iters, columns=TUNE_SEARCH_SPACE)
    trials["r2"] = tune_result.r2_scores
    trials["mae"] = tune_result.mae_scores
    trials["rmse"] = tune_result.rmse_scores
    trials["spearman"] = tune_result.spearman_scores
    trials["coverage"] = tune_result.coverage_scores
    trials["r2_utility"] = tune_result.r2_utility_scores
    trials["objective"] = tune_result.objective_scores
    display(trials.sort_values("objective", ascending=False).reset_index(drop=True))
elif RUN_IMSTE_TUNING:
    print("Tuning skipped because retweet counts are constant; continuing without target supervision.")
else:
    print("Tuning skipped by configuration; using the configured IMSTE parameters.")

Tuning skipped by configuration; using the configured IMSTE parameters.


## 4. Review or override the suggested parameters

Set any values in IMSTE_OVERRIDES. These take precedence over the tuner's suggestion before the full layout fit. `target_strength` and `target_bandwidth` control how retweet-count similarity changes graph attraction.

In [5]:
IMSTE_OVERRIDES = {
    # Examples: "target_strength": 2.0, "target_bandwidth": 0.75, "n_msts": 8
}
FINAL_IMSTE_PARAMS = {
    **IMSTE_BASE_PARAMS, **IMSTE_TUNED_PARAMS, **IMSTE_OVERRIDES,
    "random_state": RANDOM_STATE, "metric": METRIC, "verbose": True,
}
if not TARGET_HAS_VARIATION:
    FINAL_IMSTE_PARAMS["target_strength"] = 0.0
print("Regression target:", REGRESSION_TARGET_NAME)
print("Tuner best parameters:", IMSTE_TUNED_PARAMS or "(none)")
print("Final parameters after user overrides:", FINAL_IMSTE_PARAMS)

Regression target: log1p(retweet_count)
Tuner best parameters: (none)
Final parameters after user overrides: {'n_msts': 3, 'learning_rate': 0.05, 'negative_ratio': 5, 'n_epochs': 200, 'target_strength': 0.0, 'target_bandwidth': 1.0, 'gravity_strength': 0.2, 'mean_shift_iterations': 3, 'random_state': 42, 'metric': 'euclidean', 'verbose': True}


In [6]:
umap_started = time.perf_counter()
Z_umap = umap.UMAP(**UMAP_SETTINGS).fit_transform(X)
umap_seconds = time.perf_counter() - umap_started
print(f"UMAP fit: {umap_seconds / 60:.1f} min")
imste_started = time.perf_counter()
imste_estimator = IteratedMinimumSpanningTreeEmbedder(**FINAL_IMSTE_PARAMS)
if TARGET_HAS_VARIATION:
    Z_imste = imste_estimator.fit_transform(X, y=regression_target)
    fit_description = "Target-shaped IMSTE fit"
else:
    Z_imste = imste_estimator.fit_transform(X)
    fit_description = "Unsupervised IMSTE fit (constant retweet-count target)"
imste_seconds = time.perf_counter() - imste_started
print(f"{fit_description}: {imste_seconds / 60:.1f} min")

UMAP(compatibility_layout=False, init='recursive', n_epochs=200, nearest_neighbor_backend='rinnd', random_state=42, verbose=True)
Mon Nov 23 12:12:01 2026 Construct fuzzy simplicial set
Mon Nov 23 12:12:01 2026 Finding Nearest Neighbors
Mon Nov 23 12:12:01 2026 Using RINND to find nearest neighbors
Building RP forest with 8 trees...
Initializing graph from 24497 leaves...
Running NN-descent for up to 17 iterations...
Iteration 1: 795376 updates
Iteration 2: 500902 updates
Iteration 3: 158272 updates
Iteration 4: 53099 updates
Iteration 5: 19323 updates
Iteration 6: 6933 updates
Iteration 7: 2637 updates
Iteration 8: 991 updates
Converged after 8 iterations

=== Timing Breakdown ===
Forest building:     454.258ms
Leaf initialization: 338.551ms
Candidate building:   59.883ms (8 iters)
Update iterations:   756.848ms (8 iters)
Total measured:     1609.540ms
Mon Nov 23 12:12:02 2026 Finished Nearest Neighbor Search
Mon Nov 23 12:12:04 2026 Construct embedding
Mon Nov 23 12:12:04 2026 Recurs

/Users/f.costa/.venvs/py312/lib/python3.12/site-packages/sklearn/utils/extmath.py:378: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/f.costa/.venvs/py312/lib/python3.12/site-packages/sklearn/utils/extmath.py:378: RuntimeWarning: overflow encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/f.costa/.venvs/py312/lib/python3.12/site-packages/sklearn/utils/extmath.py:378: RuntimeWarning: invalid value encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/f.costa/.venvs/py312/lib/python3.12/site-packages/sklearn/utils/extmath.py:379: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Users/f.costa/.venvs/py312/lib/python3.12/site-packages/sklearn/utils/extmath.py:379: RuntimeWarning: overflow encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Users/f.costa/.venvs/py312/lib/python3.12/site-packages/sklearn/utils/extmath.py:379: RuntimeWarning: invalid value encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Users

Mon Nov 23 12:12:05 2026 Recursive initialization: coarsening 25000 nodes into about 6250 parts
Mon Nov 23 12:12:05 2026 Recursive initialization: coarsening 6250 nodes into about 1562 parts
Mon Nov 23 12:12:05 2026 Recursive initialization: coarsening 1562 nodes into about 390 parts


Init recursion depth 4:   0%|          | 0/138 [00:00<?, ?it/s]

Init recursion depth 3:   0%|          | 0/98 [00:00<?, ?it/s]

Init recursion depth 2:   0%|          | 0/70 [00:00<?, ?it/s]

... (8 outputs left out)

Multilevel refinement epochs:   0%|          | 0/3 [00:00<?, ?epoch/s]

Mon Nov 23 12:12:13 2026 Recursive initialization: refining levels
Mon Nov 23 12:12:14 2026 Recursive initialization: refining levels
Mon Nov 23 12:12:15 2026 Recursive initialization: refining levels
Mon Nov 23 12:12:15 2026 Multilevel initialization complete: 2.677s
Mon Nov 23 12:12:15 2026 Final embedding optimization


Final embedding epochs:   0%|          | 0/200 [00:00<?, ?epoch/s]

Mon Nov 23 12:12:19 2026 Final embedding optimization complete: 4.764s
Mon Nov 23 12:12:31 2026 Finished embedding
Mon Nov 23 12:12:31 2026 === Timing Breakdown ===
Mon Nov 23 12:12:31 2026 MST graph construction: 2.999s
Mon Nov 23 12:12:31 2026 Multilevel initialization: 2.677s
Mon Nov 23 12:12:31 2026 Final embedding optimization: 4.764s
Mon Nov 23 12:12:31 2026 Embedding optimization: 7.442s
Mon Nov 23 12:12:31 2026   Final MPS negative sampling and coordinate updates: 4.632s combined
Mon Nov 23 12:12:31 2026 Total measured: 23.396s
Mon Nov 23 12:12:31 2026 Multilevel initialization is included in embedding optimization.
Unsupervised IMSTE fit (constant retweet-count target): 0.4 min


## 5. Compare runtimes and display the 2D maps

In [7]:
runtime = pd.DataFrame([
    {"method": "IMSTE tuning", "seconds": tuning_seconds},
    {"method": "UMAP", "seconds": umap_seconds},
    {"method": "IMSTE", "seconds": imste_seconds},
])
runtime["runtime"] = runtime["seconds"].map(lambda x: f"{x / 60:.1f} min" if x >= 60 else f"{x:.1f} s")
display(runtime)

,method,seconds,runtime
0,IMSTE tuning,0.000000,0.0 s
1,UMAP,7.183598,7.2 s
2,IMSTE,23.398879,23.4 s


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
from ipywidgets import Dropdown, VBox, interactive_output
from imste.visualization import render_datashader_embedding

COLOR_OPTIONS = [
    "source_batch", "language", "hour", "weekday", "retweet", "retweet_count", "hashtag", "engagement",
]


def labels_for_color_by(color_by):
    timestamps = pd.to_datetime(tweets["created_at"], format="mixed", errors="coerce", utc=True)
    if color_by == "source_batch":
        color_values = tweets["source_batch"].fillna("unknown").astype(str)
    elif color_by == "language":
        color_values = tweets["lang"].fillna("unknown").astype(str)
    elif color_by == "hour":
        color_values = timestamps.dt.hour.map(
            lambda x: f"{int(x):02d}:00" if pd.notna(x) else "unknown"
        )
    elif color_by == "weekday":
        color_values = timestamps.dt.day_name().fillna("unknown")
    elif color_by == "retweet":
        color_values = tweets["is_retweet"].map(
            {True: "retweet", False: "original"}
        ).fillna("unknown")
    elif color_by == "retweet_count":
        counts = pd.to_numeric(tweets["retweet_count"], errors="coerce").fillna(0).clip(lower=0)
        color_values = pd.cut(
            counts,
            bins=[-0.5, 0.5, 1.5, 5.5, 20.5, 100.5, np.inf],
            labels=["0", "1", "2–5", "6–20", "21–100", ">100"],
        ).astype(str)
    elif color_by == "hashtag":
        hashtag_counts = tweets["top_hashtag"].value_counts()
        top_hashtags = set(hashtag_counts.head(TOP_HASHTAGS).index)
        color_values = tweets["top_hashtag"].where(
            tweets["top_hashtag"].isin(top_hashtags), "other / no hashtag"
        )
    elif color_by == "engagement":
        retweets = pd.to_numeric(tweets["retweet_count"], errors="coerce")
        favorites = pd.to_numeric(tweets["favorite_count"], errors="coerce")
        engagement = (retweets + favorites).replace([np.inf, -np.inf], np.nan).fillna(0)
        if engagement.nunique() < 2:
            value = engagement.iloc[0] if len(engagement) else 0
            color_values = pd.Series(
                f"all tweets: {value:g} engagement", index=tweets.index, dtype="string"
            )
        else:
            color_values = pd.qcut(
                np.log1p(engagement), q=5, duplicates="drop"
            ).astype(str)
    else:
        raise ValueError(f"Unknown color field: {color_by}")

    color_values = color_values.astype(str)
    counts = color_values.value_counts()
    if color_by in {"source_batch", "language", "hashtag"} and len(counts) > 20:
        keep = set(counts.head(19).index)
        color_values = color_values.where(color_values.isin(keep), "other")
    return pd.Categorical(color_values)


def plot_embeddings(color_by):
    selected_labels = labels_for_color_by(color_by)
    category_order = list(selected_labels.categories)
    if len(category_order) == 2:
        # Keep binary comparisons high-contrast; categorical order puts
        # "original" before "retweet", so retweets appear in red.
        binary_colors = ["#1f77b4", "#d62728"]  # blue, red
        color_key = dict(zip(category_order, binary_colors))
    else:
        # tab10 uses distinct hues for small category sets; tab20 adds capacity
        # when more than ten categories need colors.
        palette = plt.get_cmap("tab10" if len(category_order) <= 10 else "tab20")
        color_key = {
            name: to_hex(palette(i % palette.N))
            for i, name in enumerate(category_order)
        }
    label_values = selected_labels.astype(str)
    fig, axes = plt.subplots(1, 2, figsize=(16, 7), constrained_layout=True)
    for ax, title, coordinates in zip(axes, ("UMAP", "IMSTE"), (Z_umap, Z_imste)):
        image = render_datashader_embedding(
            coordinates[:, :2], label_values, width=1200, height=900,
            color_key=color_key, min_alpha=20, background="white", spread_px=1,
        )
        ax.imshow(image.to_pil())
        ax.set_title(f"{title} · colored by {color_by}")
        ax.set_axis_off()
    handles = [
        plt.Line2D([0], [0], marker="o", linestyle="", color=color_key[name], label=name)
        for name in category_order
    ]
    fig.legend(handles=handles, loc="outside lower center",
               ncol=min(6, len(handles)), frameon=False)
    display(fig)
    plt.close(fig)

color_by_selector = Dropdown(
    options=COLOR_OPTIONS,
    value=COLOR_BY if COLOR_BY in COLOR_OPTIONS else "source_batch",
    description="Color by:",
    layout={"width": "250px"},
)
plot_output = interactive_output(plot_embeddings, {"color_by": color_by_selector})
display(VBox([color_by_selector, plot_output]))
